In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from model import DERMANet


# deterministic seed
def set_seed(seed=42):

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    torch.backends.cudnn.deterministic = True


set_seed(42)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)


# dataset
train_dir = "dataset/train"
val_dir = "dataset/val"


transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])


train_dataset = datasets.ImageFolder(train_dir, transform)
val_dataset = datasets.ImageFolder(val_dir, transform)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)


num_classes = len(train_dataset.classes)


model = DERMANet(num_classes).to(DEVICE)


criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0001)


best_acc = 0


for epoch in range(50):

    model.train()

    correct = 0
    total = 0

    for images, labels in train_loader:

        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        _, preds = torch.max(outputs,1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)


    train_acc = 100 * correct / total


    print(f"Epoch {epoch+1}, Train Accuracy: {train_acc:.2f}")


    if train_acc > best_acc:

        best_acc = train_acc

        torch.save(model.state_dict(), "outputs/dermanet_best.pth")


print("Best accuracy:", best_acc)
